### Bielik w systemach agentowych - dodatek "Function Calling w Bieliku"

<img src="https://bielik.ai/wp-content/uploads/2024/08/Bielik_Secondary_Rabarbar-300x82.webp" width=350>

Autor: **Piotr Tynecki** | Aktualizacja: **21.11.2025**

In [ ]:
!pip install -q transformers torch accelerate sentencepiece requests

In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

In [ ]:
import os
import json
import re

from typing import Optional, Dict

import requests

### Sprawdzanie dostępności GPU 🚀

In [ ]:
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Używanie urządzenia: {device}")

In [ ]:
DEVICE = torch.device("cuda")

### Logowanie do Hugging Face 🤗

In [ ]:
from google.colab import userdata
from huggingface_hub import login

login(token=userdata.get('HF_TOKEN'))

print("Zalogowano do Hugging Face")

### Pobieranie modelu Bielik 4.5B v3.0 🦅

In [ ]:
MODEL_NAME = "speakleash/Bielik-4.5B-v3.0-Instruct"

In [ ]:
# Ładowanie tokenizera - zasady konwersji tekstu na tokeny
print("Ładowanie tokenizera...\n")
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

In [ ]:
# Ładowanie modelu z precyzją bfloat16
print("Ładowanie modelu (to może potrwać kilka minut)...")
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    dtype=torch.bfloat16,
    low_cpu_mem_usage=True,
    device_map="auto"
)

In [ ]:
print(f"[INFO] Całkowita pamięć vRAM zajęta przez model: {model.get_memory_footprint() / (1024 ** 3):.2f} GB")

### Testowa interakcja z modelem ⏱️

In [ ]:
user_prompt = "Jaki mamy dzisiaj dzień? Podaj odpowiedź w formacie DD-MM-RRRR."

In [ ]:
messages = [
    {"role": "system", "content": "Jesteś asystentem o nazwie Konfident. Odpowiadasz krótko i celnie na pytania użytkownika"},
    {"role": "user", "content": user_prompt}
]

In [ ]:
input_ids = tokenizer.apply_chat_template(
    messages,
    tokenize=True,
    add_generation_prompt=True,
    return_tensors="pt",
    padding="longest", # Ważne dla prawidłowego attention_mask
    return_attention_mask=True
).to(DEVICE)

In [ ]:
attention_mask = (
    input_ids != tokenizer.pad_token_id
).long().to(DEVICE)

In [ ]:
MAX_NEW_TOKENS = 512
TEMPERATURE = 0.7
TOP_P = 0.9

In [ ]:
with torch.no_grad():
    outputs = model.generate(
        input_ids,
        attention_mask=attention_mask,
        max_new_tokens=MAX_NEW_TOKENS,
        do_sample=True,
        temperature=TEMPERATURE,
        top_p=TOP_P,
        eos_token_id=tokenizer.eos_token_id,
        pad_token_id=tokenizer.pad_token_id,
    )

In [ ]:
response = tokenizer.decode(outputs[0], skip_special_tokens=True)
print(response)

### Function Calling 🦾 ⛅

#### [Krok 1] Kodujemy, nazywamy i opisujemy rzeczywistą funkcję

In [ ]:
def get_current_weather(city: str) -> Optional[Dict[str, str]]:
    """Sprawdza aktualną pogodę dla miasta na podstawie danych z IMGW"""

    city = city.title()

    # Aktualne dane ze stacji synoptycznych
    IMGW_API_URL = "https://danepubliczne.imgw.pl/api/data/synop"

    try:
        response = requests.get(IMGW_API_URL)
        response.raise_for_status()

        data = response.json()

        city_data = next((item for item in data if item.get("stacja") == city), None)

        if city_data:
            city_data = {
                "temperatura": f"{city_data.get("temperatura", "N/A")} °C",
                "ciśnienie": f"{city_data.get("cisnienie", "N/A")} hPa",
                "opady": f"{city_data.get("suma_opadu", "N/A")} mm",
                "prędkość_wiatru": f"{city_data.get("predkosc_wiatru", "N/A")} m/s",
                "czas_pomiaru": f"{city_data.get("data_pomiaru", "N/A")} {city_data.get("godzina_pomiaru", "N/A")}:00"

            }

            return city_data
        else:
            print(f"⚠️ Nie znaleziono danych dla miasta {city}.")

            return None
    except requests.exceptions.RequestException as e:
        print(f"❌ Wystąpił błąd podczas pobierania danych: {e}")

        return None

In [ ]:
# get_current_weather(city="warszawa")
get_current_weather(city="Białystok")
# get_current_weather(city="Suwałki")

#### [Krok 2] Tworzymy opis naszych funkcji

Struktura opisu jest narzucona. Opis musi być zrozumiały dla modelu, by wiedział w jakich okolicznościach ma skorzystać z dostępnych funkcji.

In [ ]:
tools_spec = [
    {
        "type": "function",
        "function": {
            "name": "get_current_weather", # Definicja funkcji z Kroku 1
            "description": "Sprawdza aktualną pogodę w podanym polskim mieście", # Zrozumiały opis dla modelu
            "parameters": {
                "type": "object",
                "properties": {
                    "city": {
                        "type": "string",
                        "description": "Nazwa miasta w Polsce, np. Warszawa, Kraków"
                    }
                },
                "required": ["city"]
            }
        }
    }
]

⚠️ Autorzy Bielika rekomendują by opisy funkcji i argumentów (parametr `description`) były jednak w języku angielskim. Do obsługi Function Callingu Bielik był trenowany na opisach w języku angielskim. Zobacz przykład [bielik-tools](https://github.com/speakleash/bielik-tools/blob/main/examples/tool_calling.py).

In [ ]:
# Konwertujemy na format JSON dla modelu
tools_spec_json = json.dumps(tools_spec, ensure_ascii=False)

#### [Krok 3] Łączymy prompt systemowy (system) i prompt użytkownika (user), by asystent wybrał właściwą funkcję

In [ ]:
# Zwrócmy uwagę, że nazwa miasta nie została wprost określona
user_prompt = "Jaka jest pogoda w stolicy Województwa Podlaskiego?"

In [ ]:
system_prompt = """
Jesteś pomocnym asystentem AI. Gdy potrzebujesz dodatkowych informacji,
możesz używać dostępnych funkcji. Odpowiadaj jednym zdaniem.
"""

In [ ]:
# Dołączenie do promptu systemowego listy dostępnych funkcji (ich specyfikacji)
full_prompt = f"{user_prompt}\n\nDostępne funkcje: {tools_spec_json}"

In [ ]:
# Tagowanie promptu zgodnie z szablonem modelu konwersacyjnego
formatted_prompt = f"<|im_start|> system\n{system_prompt}<|im_end|>\n<|im_start|> user\n{full_prompt}<|im_end|>\n<|im_start|> assistant\n"
formatted_prompt

In [ ]:
inputs = tokenizer(
    formatted_prompt,
    return_tensors="pt"
).to(DEVICE)

In [ ]:
outputs = model.generate(
    inputs["input_ids"],
    max_new_tokens=500
)

In [ ]:
raw_response = tokenizer.decode(
    outputs[0],
    skip_special_tokens=False
)

print(raw_response)

#### [Krokt 4] Wydobywamy wybrane narzędzie (tool_call) i przekazujemy jego wynik działania ponownie do asystenta

In [ ]:
def extract_function_calling(text: str) -> Optional[str]:
    """Wyodrębnia wywołanie funkcji z tagu."""

    # Szukamy wzorca JSON
    pattern = r"<tool_call>\s*(.*?)\s*</tool_call>"
    match = re.search(pattern, text, re.DOTALL)

    if match:
        try:
            tool_call_execution = json.loads(match.group(1))

            # Konwertujemy "name" na "function" dla spójności
            if "name" in tool_call_execution:
                tool_call_execution["function"] = tool_call_execution["name"]

            return tool_call_execution
        except Exception as e:
            print(f"❌ Wystąpił błąd podczas wyodrębnia funkcji: {e}")

    return None

In [ ]:
tool_selection_results = extract_function_calling(raw_response)

In [ ]:
print("\nWyodrębnione wywołanie funkcji:")
print(json.dumps(
    tool_selection_results,
    indent=4,
    ensure_ascii=False
) if tool_selection_results else "Nie znaleziono wywołania funkcji")

In [ ]:
if tool_selection_results and (
    "function" in tool_selection_results and "arguments" in tool_selection_results # Tylko w przypadku gdy funkcja ma wymagane argumenty
):
    function_name = tool_selection_results["function"]
    function_args = tool_selection_results["arguments"]

    # Próbujemy wykonać funkcję
    try:
        # Używamy eval() aby wywołać funkcję z argumentami
        tool_prompt = eval(f"{function_name}(**{function_args})")

        print(f"\nWykonuję funkcję: {function_name}")
        print(f"Przekazywane argumenty: {json.dumps(function_args, ensure_ascii=False)}")
    except Exception as e:
        tool_prompt = {"function": function_name, "error": str(e)}
        print(f"\nBłąd: {str(e)}")
else:
    tool_prompt = None
    print("\nNie można wykonać funkcji - brak poprawnego wywołania.")

In [ ]:
tool_prompt

In [ ]:
# Wyświetlamy wynik, jeśli jest
if tool_prompt is not None:
    print("\nWynik funkcji:")
    print(json.dumps(tool_prompt, indent=4, ensure_ascii=False))

In [ ]:
if tool_prompt is not None:
    # Tworzymy nowy prompt zawierający całą historię
    full_history_prompt = (
        # Etap 1: Prompt systemowy + prompt użytkownika + prompt asystenta
        f"{raw_response}\n"
        # Etap 2: Prompt tool + prompt asystenta
        f"<|im_start|> tool\n"
        f"{json.dumps(tool_prompt, ensure_ascii=False)}<|im_end|>\n"
        # Etap 3: Sformatowana odpowiedź
        f"<|im_start|> assistant\n"
    )

    # Tokenizacja
    final_inputs = tokenizer(
        full_history_prompt,
        return_tensors="pt"
    ).to(DEVICE)

    # Generowanie odpowiedzi
    final_outputs = model.generate(
        final_inputs["input_ids"],
        max_new_tokens=500
    )

    # Dekodowanie odpowiedzi
    final_response = tokenizer.decode(
        final_outputs[0],
        skip_special_tokens=False
    )
    print("\nOstateczna odpowiedź modelu (z wynikiem funkcji):\n")
    print(final_response)

    print("\n\n-------------------------------\n")

    # Wyodrębniamy tylko ostatnią odpowiedź asystenta
    pattern = r"<\|im_start\|\>\s+assistant\n(.*?)<\|im_end\|>"
    matches = re.findall(pattern, final_response, re.DOTALL)

    if matches:
        last_assistant = matches[-1].strip() # Bierzemy ostatnie dopasowanie
        print("\nOstateczna odpowiedź (oczyszczona):\n")
        print(last_assistant)
    else:
        print("\nNie znaleziono odpowiedzi asystenta")